# Live demo: next-hour high-demand alert

**Input:** `data/ed_hourly_clean.csv` (cleaned hourly data from the shared pipeline). No other file is needed.
**Model:** Logistic Regression without class weights (calibrated probabilities); alert cutoff chosen on the 2016 validation year.
**Output:** for any hour, the probability of more than 8 arrivals, the alert decision, and what actually happened.

In [ ]:
# ===== Cell 0 (Colab only): get the repository and its data
!git clone -q https://github.com/srijataduri/ED-High-Demand-Prediction.git
%cd ED-High-Demand-Prediction

In [ ]:
# ===== Cell 1: train once and save the demo model =====
import json, joblib, numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, average_precision_score

import os
DATA = next(p for p in ("data/ed_hourly_clean.csv", "ed_hourly_clean.csv",
                        "/content/ED-High-Demand-Prediction/data/ed_hourly_clean.csv") if os.path.exists(p))
print("Input data:", DATA)
CAL = ["hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos", "month_sin", "month_cos", "is_weekend", "is_holiday"]
LAGS = ["lag_1", "lag_2", "lag_3", "lag_24", "lag_168", "rolling_mean_3", "rolling_mean_6",
        "rolling_mean_24", "rolling_std_24", "rolling_max_24", "same_hour_mean_7d"]

df = pd.read_csv(DATA, parse_dates=["datetime"])
lab = df[df.high_demand.notna()]
tr, va, te = (lab[lab.split == s] for s in ("train", "validation", "test"))

model = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), CAL + LAGS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["hour", "day_of_week"])])),
    ("clf", LogisticRegression(C=0.03, max_iter=3000))])          # no class weights -> calibrated probabilities
model.fit(tr, tr.high_demand.astype(int))

# alert cutoff chosen on the 2016 validation year only (max F1)
p, r, t = precision_recall_curve(va.high_demand, model.predict_proba(va)[:, 1])
f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
CUTOFF = float(t[np.nanargmax(f1)])

joblib.dump({"model": model, "cutoff": CUTOFF, "threshold_arrivals": 8}, "ed_alert_model.joblib")
print(f"Saved ed_alert_model.joblib | cutoff = {CUTOFF:.3f} | "
      f"test PR-AUC = {average_precision_score(te.high_demand, model.predict_proba(te)[:, 1]):.3f}")

In [ ]:
# ===== Cell 2: live demo — predict the next hour =====
import joblib, pandas as pd
bundle = joblib.load("ed_alert_model.joblib")
df = pd.read_csv(DATA, parse_dates=["datetime"])

def next_hour_alert(when):
    """Score the hour starting at `when`, using only information available before it."""
    row = df[df.datetime == pd.Timestamp(when)]
    if row.empty or row.lag_1.isna().all():
        return print(f"{when}: no usable input (missing data)")
    prob = bundle["model"].predict_proba(row)[0, 1]
    alert = prob >= bundle["cutoff"]
    r = row.iloc[0]
    print(f"Hour starting   : {r.datetime:%a %Y-%m-%d %H:%M}")
    print(f"Inputs (past)   : last hour = {r.lag_1:.0f}, same hour last week = {r.lag_168:.0f}, "
          f"7-day same-hour mean = {r.same_hour_mean_7d:.1f}")
    print(f"P(> {bundle['threshold_arrivals']} arrivals): {prob:.2f}   (alert cutoff {bundle['cutoff']:.2f})")
    print(f"Decision        : {'ALERT - expect high demand' if alert else 'no alert'}")
    if pd.notna(r.arrivals):
        print(f"What happened   : {r.arrivals:.0f} arrivals -> {'high demand' if r.arrivals > 8 else 'normal'}")
    print()

next_hour_alert("2017-03-13 17:00")   # Monday 5 pm
next_hour_alert("2017-03-14 04:00")   # Tuesday early morning

In [ ]:
# ===== Cell 3 (optional): a whole day as the ED manager would see it =====
day = df[df.datetime.dt.date == pd.Timestamp("2017-03-13").date()].dropna(subset=["lag_1"]).copy()
day["prob"] = bundle["model"].predict_proba(day)[:, 1]
day["alert"] = day.prob >= bundle["cutoff"]
print(day[["datetime", "prob", "alert", "arrivals"]].assign(prob=lambda x: x.prob.round(2)).to_string(index=False))